# 08 Damage-Informed Tide-Gage Placement

> **Stage Contract**
>
> Requires: completed SFINCS `run_outputs`, Event Catalog weights from `03`, and FIAT per-event damage outputs from `07`.
>
> Produces: candidate deployable water-level sensor scores, a selected sensor network, candidate/event response tables, and review figures under `data/sfincs/tide_gauges`.
>
> Method: this is a diagnostic sensor-placement screen. It ranks locations whose local modeled flood response best predicts or explains FIAT damages across the catalog; it does not claim sensors themselves reduce damage. Existing SFINCS runup-gauge transects are computational model outputs and candidate seeds, not installed physical instruments.

In [ ]:
import sys, json
from pathlib import Path

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parent
repo_root = location_root.parents[1]
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
config_yaml = location_root / "config.yaml"

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from sfincs_runs.config import load_runtime
from sfincs_runs import diagnostics as sfincs_diagnostics
from sfincs_runs import tide_gauges
from fiat_runs.config import fiat_paths
from fiat_runs import diagnostics as fiat_diagnostics
from fiat_runs import risk as fiat_risk

config, paths = load_runtime(config_yaml)
paths = fiat_paths(paths)
pd.set_option("display.max_columns", 100)

MODEL_CRS = tide_gauges.MODEL_CRS
SENSOR_COUNT = 3
CANDIDATE_TOP_N_BUILDING_CLUSTERS = 30
SAMPLE_RADIUS_M = 120.0
MIN_SELECTED_DISTANCE_M = 350.0
DAMAGE_COLUMN = "total_damage"
DEPTH_PROBABILITY_THRESHOLD_FT = 1.0

out_root = paths["location_data_root"] / "sfincs" / "tide_gauges"
fig_root = out_root / "figures"
out_root.mkdir(parents=True, exist_ok=True)
fig_root.mkdir(parents=True, exist_ok=True)

catalog_csv = paths["design_outputs_root"] / "catalog" / "event_catalog.csv"
metadata_json = paths["design_outputs_root"] / "catalog" / "catalog_risk_metadata.json"
per_event_damage_csv = paths["fiat_risk_root"] / "per_event_damage.csv"
model_root = paths["fiat_model_root"]

## Step 0 - Input audit

Load the completed SFINCS flood catalogue, Event Catalog probability weights, and FIAT damage outputs. The SFINCS coverage and FIAT damage coverage are reported separately so partial FIAT coverage is not silently treated as a full-catalog risk result.

In [ ]:
if not catalog_csv.exists() or not metadata_json.exists():
    raise FileNotFoundError("Missing Event Catalog or catalog_risk_metadata.json. Run 03_build_event_catalog first.")

runs = sfincs_diagnostics.completed_sfincs_runs(paths["storage_root"])
if runs.empty:
    raise RuntimeError("No completed SFINCS run_outputs found. Run 05 and the SFINCS jobs first.")

weights = fiat_risk.load_catalog_weights(catalog_csv)
total_rate = fiat_risk.total_rate_from_metadata(metadata_json)

if per_event_damage_csv.exists():
    per_event_damage = pd.read_csv(per_event_damage_csv)
    if "design_scenario" not in per_event_damage:
        per_event_damage["design_scenario"] = "base"
else:
    per_event_damage = pd.DataFrame(columns=["event_id", "design_scenario", DAMAGE_COLUMN])

event_outcomes = sfincs_diagnostics.event_outcome_table(
    runs,
    catalog_csv,
    weights,
    total_rate,
    per_event_damage,
)

damage_mask = event_outcomes[DAMAGE_COLUMN].notna() if DAMAGE_COLUMN in event_outcomes else pd.Series(False, index=event_outcomes.index)
damage_outcomes = event_outcomes[damage_mask].copy()
sfincs_coverage = sfincs_diagnostics.outcome_coverage(event_outcomes, weights)
damage_coverage = sfincs_diagnostics.outcome_coverage(damage_outcomes, weights) if len(damage_outcomes) else pd.Series(
    {
        "completed_outcome_events": 0,
        "catalog_weighted_events": len(weights),
        "covered_probability_weight": 0.0,
        "catalog_probability_weight": float(pd.to_numeric(weights["probability_weight"], errors="coerce").sum()),
        "weight_coverage": 0.0,
    },
    name="fiat_damage_probability_coverage",
)

audit = pd.Series(
    {
        "completed_sfincs_events": len(runs),
        "fiat_damage_events": len(damage_outcomes),
        "catalog_weighted_events": len(weights),
        "sfincs_weight_coverage": sfincs_coverage["weight_coverage"],
        "fiat_damage_weight_coverage": damage_coverage["weight_coverage"],
        "total_rate_per_year": total_rate,
        "per_event_damage_csv": str(per_event_damage_csv),
    },
    name="tide_gage_placement_inputs",
)
display(audit)
display(damage_outcomes[[c for c in ["event_id", "design_scenario", "storm_type", "severity_band", "probability_weight", "annual_rate", DAMAGE_COLUMN] if c in damage_outcomes]].head())

## Step 1 - Existing SFINCS runup transects

These transects are model-output features used by the wave-coupled SFINCS build. They are useful candidate seeds, but they are not physical tide gages or validation sensors.

In [ ]:
transects = tide_gauges.load_runup_transects(location_root / "sfincs.yaml", crs=MODEL_CRS)
runup_candidates = tide_gauges.candidate_points_from_runup_transects(transects, crs=MODEL_CRS)

display(transects[["gauge", "candidate_source"]] if not transects.empty else pd.DataFrame(columns=["gauge", "candidate_source"]))

fig, ax = plt.subplots(figsize=(8, 7))
if not transects.empty:
    transects.plot(ax=ax, color="#3182bd", linewidth=2.4, label="SFINCS runup transect", zorder=3)
if not runup_candidates.empty:
    runup_candidates.plot(ax=ax, color="#31a354", markersize=70, edgecolor="white", linewidth=0.8, label="candidate seed", zorder=4)
try:
    ctx.add_basemap(ax, crs=MODEL_CRS, source=ctx.providers.OpenStreetMap.HOT, attribution_size=7)
except Exception as exc:
    ax.text(0.01, 0.01, f"Basemap unavailable: {exc}", transform=ax.transAxes, fontsize=8)
ax.set_title("Existing computational SFINCS runup transects")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(fig_root / "existing_runup_transects.png", dpi=180)
plt.show()

## Step 2 - FIAT building-risk surface

Use the existing building annualized-risk product when available. If `07_risk_fiat.ipynb` has not exported it yet, aggregate the available per-event FIAT `spatial.gpkg` outputs here so sensor candidates can still be seeded from damaging structure clusters.

In [ ]:
building_risk_gpkg = paths["fiat_risk_root"] / "building_annualized_risk.gpkg"
building_risk_csv = paths["fiat_risk_root"] / "building_annualized_risk.csv"
exposure_csv = model_root / "exposure" / "exposure.csv"

if building_risk_gpkg.exists():
    building_risk = gpd.read_file(building_risk_gpkg).to_crs(MODEL_CRS)
elif len(damage_outcomes):
    building_risk = fiat_diagnostics.aggregate_building_risk(
        paths["fiat_risk_root"],
        damage_outcomes,
        scenario="base",
        exposure_csv=exposure_csv,
    ).to_crs(MODEL_CRS)
else:
    building_risk = gpd.GeoDataFrame(columns=["object_id", "annual_damage", "damage_aep", "geometry"], geometry="geometry", crs=MODEL_CRS)

risk_summary = pd.Series(
    {
        "building_risk_assets": len(building_risk),
        "positive_annual_damage_assets": int((pd.to_numeric(building_risk.get("annual_damage"), errors="coerce").fillna(0) > 0).sum()) if len(building_risk) else 0,
        "total_annual_damage_represented": float(pd.to_numeric(building_risk.get("annual_damage"), errors="coerce").fillna(0).sum()) if len(building_risk) else 0.0,
        "source": "existing building_annualized_risk.gpkg" if building_risk_gpkg.exists() else "aggregated available FIAT spatial.gpkg outputs",
    },
    name="building_risk_surface",
)
display(risk_summary)

top_cols = [c for c in ["object_id", "annual_damage", "damage_aep", "max_event_damage", "aggregation_label:Census Blockgroup"] if c in building_risk]
display(building_risk.sort_values("annual_damage", ascending=False)[top_cols].head(10) if len(building_risk) and "annual_damage" in building_risk else pd.DataFrame(columns=top_cols))

## Step 3 - Candidate deployable sensor points

Candidate points combine the computational runup transect midpoints with FIAT damage-cluster seeds. This is an installability screen, not a final engineering siting decision.

In [ ]:
risk_candidates = tide_gauges.candidate_points_from_building_risk(
    building_risk,
    top_n=CANDIDATE_TOP_N_BUILDING_CLUSTERS,
    metric="annual_damage",
    min_distance_m=SAMPLE_RADIUS_M,
    crs=MODEL_CRS,
)

candidate_frames = [gdf for gdf in [runup_candidates, risk_candidates] if gdf is not None and not gdf.empty]
if not candidate_frames:
    raise RuntimeError("No candidate sensor points could be generated from runup transects or FIAT risk clusters.")

candidates = gpd.GeoDataFrame(pd.concat(candidate_frames, ignore_index=True), geometry="geometry", crs=MODEL_CRS)
candidates = candidates.drop_duplicates("candidate_id").reset_index(drop=True)

display(candidates[["candidate_id", "candidate_source", "candidate_label", "nearby_annual_damage", "x", "y"]].head(20))

fig, ax = plt.subplots(figsize=(9, 8))
seed_scores = candidates.copy()
seed_scores["score"] = 0.0
tide_gauges.plot_candidate_scores(seed_scores, building_risk=building_risk, ax=ax, title="Candidate sensor seeds over FIAT annualized damage")
fig.tight_layout()
fig.savefig(fig_root / "candidate_sensor_seeds.png", dpi=180)
plt.show()

## Step 4 - Sample the flood catalogue at candidate points

For each completed SFINCS event, sample the local peak flood depth near each candidate. The joined response table keeps missing FIAT damage rows as missing, not zero, so score coverage remains honest.

In [ ]:
samples = tide_gauges.sample_sfincs_at_candidates(
    runs,
    candidates,
    sample_radius_m=SAMPLE_RADIUS_M,
)
response = tide_gauges.candidate_event_response_table(samples, event_outcomes, damage_column=DAMAGE_COLUMN)
response_path = out_root / "candidate_event_response.csv"
response.to_csv(response_path, index=False)

response_summary = response.groupby("candidate_id").agg(
    events_sampled=("event_id", "nunique"),
    max_depth_ft=("nearby_max_depth_ft", "max"),
    mean_depth_ft=("nearby_max_depth_ft", "mean"),
    fiat_damage_events=(DAMAGE_COLUMN, lambda s: int(pd.to_numeric(s, errors="coerce").notna().sum())),
).reset_index()
display(response_summary.head(20))
print(f"Wrote {response_path}")

## Step 5 - Score and select the sensor network

The score blends weighted event-damage association, high-damage event discrimination, and nearby annualized building damage. Selected locations are spaced by `MIN_SELECTED_DISTANCE_M` to avoid choosing redundant points in the same cluster.

In [ ]:
scores = tide_gauges.score_sensor_candidates(
    response,
    candidates,
    signal_column="nearby_max_depth_ft",
    damage_column=DAMAGE_COLUMN,
    weight_column="probability_weight",
)
selected = tide_gauges.greedy_sensor_selection(
    scores,
    sensor_count=SENSOR_COUNT,
    min_distance_m=MIN_SELECTED_DISTANCE_M,
)
scores = tide_gauges.mark_selected_candidates(scores, selected)

score_cols = [
    "candidate_id", "candidate_source", "x", "y", "score",
    "weighted_damage_correlation", "high_damage_auc_like_score",
    "nearby_annual_damage", "represented_damage_share", "selected_rank",
]

scores[score_cols].to_csv(out_root / "candidate_sensor_scores.csv", index=False)
selected[score_cols].to_csv(out_root / "selected_sensor_network.csv", index=False)
scores.to_file(out_root / "candidate_sensor_scores.gpkg", driver="GPKG")
selected.to_file(out_root / "selected_sensor_network.gpkg", driver="GPKG")

display(scores[score_cols].head(15))
display(selected[score_cols])

## Step 6 - Placement maps

Review candidate scores, selected sensor points, existing computational runup transects, and the damage/flood surfaces that informed the ranking.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
tide_gauges.plot_candidate_scores(scores, building_risk=building_risk, ax=ax, title="Candidate sensor score over FIAT annualized damage")
fig.tight_layout()
fig.savefig(fig_root / "candidate_sensor_scores.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9, 8))
tide_gauges.plot_selected_sensor_network(
    scores,
    selected,
    building_risk=building_risk,
    transects=transects,
    ax=ax,
    title="Selected deployable water-level sensor network",
)
fig.tight_layout()
fig.savefig(fig_root / "selected_sensor_network.png", dpi=180)
plt.show()

## Step 7 - Selected sensors over flood-depth probability

Compute a catalog-weighted annual exceedance probability map for the review threshold and overlay the selected sensor network. This uses the Event Catalog annual rates directly and does not renormalize partial coverage.

In [ ]:
depth_probability_path = out_root / "flood_depth_probability_for_sensor_review.nc"
if depth_probability_path.exists():
    depth_probability = xr.open_dataset(depth_probability_path)
else:
    depth_probability = sfincs_diagnostics.catalog_depth_probability(
        runs,
        event_outcomes,
        thresholds_ft=(DEPTH_PROBABILITY_THRESHOLD_FT,),
    )
    depth_probability.to_netcdf(depth_probability_path)

fig, ax = plt.subplots(figsize=(9, 8))
_, mesh = sfincs_diagnostics.plot_depth_probability(
    depth_probability,
    DEPTH_PROBABILITY_THRESHOLD_FT,
    ax=ax,
    basemap_style="osm",
    title=f"Selected sensors over P(depth > {DEPTH_PROBABILITY_THRESHOLD_FT:g} ft)",
    crs=MODEL_CRS,
)
if not selected.empty:
    selected.to_crs(MODEL_CRS).plot(ax=ax, markersize=170, color="#31a354", edgecolor="white", linewidth=1.2, zorder=5)
    for row in selected.to_crs(MODEL_CRS).itertuples(index=False):
        ax.annotate(str(int(row.selected_rank)), xy=(row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords="offset points", color="white", fontweight="bold", zorder=6)
fig.tight_layout()
fig.savefig(fig_root / "selected_sensors_over_depth_probability.png", dpi=180)
plt.show()
print(f"Depth probability product: {depth_probability_path}")

## Step 8 - Damage-response scatter panels

For the selected sensors, inspect whether local modeled depth separates high-damage events. Storm-type colors are shown when the Event Catalog has `storm_type`. Marker size is proportional to probability weight.

In [ ]:
if selected.empty:
    print("No selected sensors to plot.")
else:
    n = len(selected)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4.8), squeeze=False)
    for ax, row in zip(axes.ravel(), selected.itertuples(index=False)):
        tide_gauges.plot_candidate_damage_response(
            response,
            row.candidate_id,
            signal_column="nearby_max_depth_ft",
            damage_column=DAMAGE_COLUMN,
            storm_type_column="storm_type",
            ax=ax,
            title=f"Rank {int(row.selected_rank)} - {row.candidate_id}",
        )
    fig.tight_layout()
    fig.savefig(fig_root / "selected_sensor_damage_response.png", dpi=180)
    plt.show()

## Step 9 - Output receipt

In [ ]:
receipt = {
    "method": "damage-informed water-level sensor placement",
    "sensor_count": SENSOR_COUNT,
    "candidate_count": int(len(candidates)),
    "selected_count": int(len(selected)),
    "sample_radius_m": SAMPLE_RADIUS_M,
    "min_selected_distance_m": MIN_SELECTED_DISTANCE_M,
    "completed_sfincs_events": int(len(runs)),
    "fiat_damage_events": int(len(damage_outcomes)),
    "sfincs_weight_coverage": float(sfincs_coverage["weight_coverage"]),
    "fiat_damage_weight_coverage": float(damage_coverage["weight_coverage"]),
    "outputs": {
        "candidate_event_response": str(out_root / "candidate_event_response.csv"),
        "candidate_sensor_scores": str(out_root / "candidate_sensor_scores.csv"),
        "selected_sensor_network": str(out_root / "selected_sensor_network.csv"),
        "candidate_sensor_scores_gpkg": str(out_root / "candidate_sensor_scores.gpkg"),
        "selected_sensor_network_gpkg": str(out_root / "selected_sensor_network.gpkg"),
        "figures": str(fig_root),
    },
    "interpretation": "Diagnostic/predictive placement screen, not causal damage-reduction attribution.",
}
receipt_path = out_root / "sensor_placement_receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
display(pd.Series(receipt, name="sensor_placement_receipt"))
print(f"Wrote {receipt_path}")